Regression with budgets - two approaches - as variable, b as a bracket seperator where we build multiple models

In [1]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import entropy
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, classification_report
from sklearn.preprocessing import OneHotEncoder

In [2]:
PROJECT_ROOT = Path.cwd().parents[0]

# Load admissions data.
df_admissions = pd.read_csv(PROJECT_ROOT / "Data" / "Lumiere_Country_Full_Backup.csv")

# Load content clusters.
df_clusters = pd.read_csv(
    PROJECT_ROOT / "outputs" / "text_clustering"
    / "results_content_only" / "clustered_films_k6.csv"
)

# Load budget data (Both + Budget only sheets).
budget_both = pd.read_excel(PROJECT_ROOT / "Data" / "IMDB_financials.xlsx", sheet_name="Both")
budget_only = pd.read_excel(PROJECT_ROOT / "Data" / "IMDB_financials.xlsx", sheet_name="Budget only")
df_budget = pd.concat([budget_both, budget_only], ignore_index=True)
df_budget = df_budget[["titleId", "budget_eur"]].dropna(subset=["budget_eur"])

# Merge admissions + clusters (by row index).
assert len(df_admissions) == len(df_clusters), "Row count mismatch!"
df = df_admissions.copy()
df["cluster"] = df_clusters["cluster"].values

# Merge with budget (by titleId).
df = df.merge(df_budget, on="titleId", how="inner")

print(f"Admissions films: {len(df_admissions)}")
print(f"Films with budget: {len(df_budget)}")
print(f"Merged (admissions + clusters + budget): {len(df)}")
print(f"\nBudget distribution:")
print(df["budget_eur"].describe().round(0))

Admissions films: 2000
Films with budget: 1107
Merged (admissions + clusters + budget): 1107

Budget distribution:
count         1107.0
mean      19342943.0
std       28631872.0
min              0.0
25%        2634300.0
50%        9200000.0
75%       23086125.0
max      230000000.0
Name: budget_eur, dtype: float64


In [3]:
def parse_country_admissions(value):
    if pd.isna(value) or value == "Not Found":
        return {}
    try:
        return json.loads(value.replace('""', '"'))
    except (json.JSONDecodeError, TypeError):
        return {}

# Expand to long format.
records = []
for idx, row in df.iterrows():
    admissions = parse_country_admissions(row["country_admissions"])
    for country, count in admissions.items():
        if country.startswith("Total"):
            continue
        records.append({
            "titleId": row["titleId"],
            "originalTitle": row["originalTitle"],
            "releaseYear": row["releaseYear"],
            "mainCountry": row["mainCountry"],
            "firstLanguage": row["firstLanguage"],
            "genres": row["genres"],
            "keywords": row["keywords"],
            "cluster": row["cluster"],
            "budget_eur": row["budget_eur"],
            "country": country,
            "admissions": count
        })

df_long = pd.DataFrame(records)

# Remove small countries.
MIN_FILMS = 50
country_film_counts = df_long.groupby("country")["titleId"].nunique()
small_countries = country_film_counts[country_film_counts < MIN_FILMS].index.tolist()
df_long = df_long[~df_long["country"].isin(small_countries)].copy()

# Handle GB/IE overlap.
films_with_gbie = set(df_long[df_long["country"] == "GB_IE"]["titleId"])
mask_redundant = (
    df_long["country"].isin(["GB", "IE"]) &
    df_long["titleId"].isin(films_with_gbie)
)
df_long = df_long[~mask_redundant].copy()

# Domestic flag.
def is_domestic(row):
    origin = str(row["mainCountry"]).strip().upper()
    market = str(row["country"]).strip().upper()
    if market == origin:
        return True
    if market == "GB_IE" and origin in ("GB", "IE"):
        return True
    return False

df_long["is_domestic"] = df_long.apply(is_domestic, axis=1)

# Log z-scores.
df_long["log_admissions"] = np.log1p(df_long["admissions"])
df_long["admissions_zscore"] = (
    df_long.groupby("country")["log_admissions"]
    .transform(lambda x: (x - x.mean()) / x.std())
)

# Log budget.
df_long["log_budget"] = np.log1p(df_long["budget_eur"])

print(f"Long format: {len(df_long)} rows")
print(f"Films: {df_long['titleId'].nunique()}")
print(f"Countries: {df_long['country'].nunique()}")

Long format: 15870 rows
Films: 1064
Countries: 35


In [5]:
def parse_keywords(value):
    if pd.isna(value) or value == "":
        return []
    return [k.strip().lower() for k in str(value).split(",") if k.strip()]

def parse_genres(value):
    if pd.isna(value) or value == "":
        return []
    return [g.strip().lower() for g in str(value).split(",") if g.strip()]

df_long["keywords_list"] = df_long["keywords"].apply(parse_keywords)
df_long["genres_list"] = df_long["genres"].apply(parse_genres)

# Keyword frequency — need 50+ films.
keyword_film_counts = (
    df_long.drop_duplicates("titleId")
    .explode("keywords_list")
    ["keywords_list"].value_counts()
)
valid_keywords = set(keyword_film_counts[keyword_film_counts >= 50].index)
print(f"Keywords with 50+ films: {len(valid_keywords)}")

# Compute keyword × country mean z-scores.
df_kw = df_long.explode("keywords_list")
df_kw = df_kw[df_kw["keywords_list"].isin(valid_keywords)].copy()

kw_country = (
    df_kw.groupby(["keywords_list", "country"])["admissions_zscore"]
    .agg(["mean", "count"])
    .reset_index()
)
kw_country = kw_country[kw_country["count"] >= 10].copy()

# Cross-country variance per keyword.
kw_variance = (
    kw_country.groupby("keywords_list")["mean"]
    .agg(["std", "count"])
    .rename(columns={"count": "n_countries"})
)
kw_variance = kw_variance[kw_variance["n_countries"] >= 15].sort_values("std", ascending=False)

print(f"Keywords with enough cross-country data: {len(kw_variance)}")
print(f"\nTop 15 most divisive keywords:")
print(kw_variance.head(15)[["std", "n_countries"]].round(3).to_string())

Keywords with 50+ films: 470
Keywords with enough cross-country data: 470

Top 15 most divisive keywords:
                    std  n_countries
keywords_list                       
blockbuster       0.474           32
villain           0.324           33
hero              0.312           32
british           0.301           35
evil              0.287           32
evil man          0.283           32
cult film         0.275           35
underwear         0.275           28
action hero       0.269           34
writer            0.267           31
cruelty           0.264           33
laughter          0.264           33
femme fatale      0.255           34
transformation    0.255           32
good versus evil  0.253           34


In [6]:
# Tier 1: All keywords appearing in 10+ films — base features.
MIN_KW_FREQ_BASE = 10
valid_keywords_base = set(keyword_film_counts[keyword_film_counts >= MIN_KW_FREQ_BASE].index)
print(f"Tier 1 (base features): {len(valid_keywords_base)} keywords (10+ films)")

# Tier 2: Top 30 most divisive — get keyword × market interactions.
TOP_KW_INTERACTIONS = 30
top_divisive_kw = kw_variance.head(TOP_KW_INTERACTIONS).index.tolist()
print(f"Tier 2 (with market interactions): {len(top_divisive_kw)} keywords")
print(f"  Examples: {', '.join(top_divisive_kw[:10])}")

# How many total features will this create?
n_markets = df_long["country"].nunique()
print(f"\nEstimated feature counts:")
print(f"  Base keyword features: {len(valid_keywords_base)}")
print(f"  Keyword × market interactions: {len(top_divisive_kw)} × {n_markets} = {len(top_divisive_kw) * n_markets} (before sparsity filtering)")

Tier 1 (base features): 3203 keywords (10+ films)
Tier 2 (with market interactions): 30 keywords
  Examples: blockbuster, villain, hero, british, evil, evil man, cult film, underwear, action hero, writer

Estimated feature counts:
  Base keyword features: 3203
  Keyword × market interactions: 30 × 35 = 1050 (before sparsity filtering)


In [7]:
# Target.
y = df_long["admissions_zscore"].values

# Block 1: Market country.
market_dummies = pd.get_dummies(df_long["country"], prefix="market")

# Block 2: Origin country.
origin_dummies = pd.get_dummies(df_long["mainCountry"], prefix="origin")

# Block 3: Language.
lang_dummies = pd.get_dummies(df_long["firstLanguage"], prefix="lang")

# Block 4: Content cluster.
cluster_dummies = pd.get_dummies(df_long["cluster"], prefix="cluster")

# Block 5: Domestic flag.
domestic_flag = df_long["is_domestic"].astype(int).to_frame("is_domestic")

# Block 6: Genres.
all_genres = set()
for g_list in df_long["genres_list"]:
    all_genres.update(g_list)

genre_dummies = pd.DataFrame(0, index=df_long.index, columns=[f"genre_{g}" for g in sorted(all_genres)])
for i, g_list in enumerate(df_long["genres_list"]):
    for g in g_list:
        col = f"genre_{g}"
        if col in genre_dummies.columns:
            genre_dummies.iloc[i, genre_dummies.columns.get_loc(col)] = 1

# Block 7: Keywords (Tier 1 — all with 10+ films).
kw_cols = sorted([f"kw_{k}" for k in valid_keywords_base])
keyword_features = pd.DataFrame(0, index=df_long.index, columns=kw_cols)
for i, kw_list in enumerate(df_long["keywords_list"]):
    for kw in kw_list:
        col = f"kw_{kw}"
        if col in keyword_features.columns:
            keyword_features.iloc[i, keyword_features.columns.get_loc(col)] = 1

# Block 8: Log budget.
budget_feature = df_long[["log_budget"]].copy()

# Block 9: Cluster × domestic interaction.
cluster_domestic_interaction = pd.DataFrame(
    cluster_dummies.values * domestic_flag.values,
    columns=[f"{c}_x_domestic" for c in cluster_dummies.columns],
    index=df_long.index
)

print(f"Market dummies:    {market_dummies.shape}")
print(f"Origin dummies:    {origin_dummies.shape}")
print(f"Language dummies:  {lang_dummies.shape}")
print(f"Cluster dummies:   {cluster_dummies.shape}")
print(f"Domestic flag:     {domestic_flag.shape}")
print(f"Genre dummies:     {genre_dummies.shape}")
print(f"Keyword features:  {keyword_features.shape}")
print(f"Budget feature:    {budget_feature.shape}")
print(f"Cluster×domestic:  {cluster_domestic_interaction.shape}")

Market dummies:    (15870, 35)
Origin dummies:    (15870, 25)
Language dummies:  (15870, 24)
Cluster dummies:   (15870, 6)
Domestic flag:     (15870, 1)
Genre dummies:     (15870, 22)
Keyword features:  (15870, 3203)
Budget feature:    (15870, 1)
Cluster×domestic:  (15870, 6)


In [8]:
MIN_NONZERO = 20

def drop_sparse_cols(df, min_nonzero=MIN_NONZERO):
    keep = df.columns[(df != 0).sum(axis=0) >= min_nonzero]
    return df[keep]

# Language × market.
lang_vals = lang_dummies.values
market_vals = market_dummies.values
lang_market_interaction = pd.DataFrame(
    (lang_vals[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{l}_x_{m}" for l in lang_dummies.columns for m in market_dummies.columns],
    index=df_long.index
)
lang_market_trimmed = drop_sparse_cols(lang_market_interaction)

# Origin × market.
origin_vals = origin_dummies.values
origin_market_interaction = pd.DataFrame(
    (origin_vals[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{o}_x_{m}" for o in origin_dummies.columns for m in market_dummies.columns],
    index=df_long.index
)
origin_market_trimmed = drop_sparse_cols(origin_market_interaction)

# Cluster × market.
cluster_vals = cluster_dummies.values
cluster_market_interaction = pd.DataFrame(
    (cluster_vals[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{c}_x_{m}" for c in cluster_dummies.columns for m in market_dummies.columns],
    index=df_long.index
)
cluster_market_trimmed = drop_sparse_cols(cluster_market_interaction)

# Keyword × market (Tier 2 — top divisive only).
kw_tier2_cols = [f"kw_{k}" for k in top_divisive_kw if f"kw_{k}" in keyword_features.columns]
kw_tier2_matrix = keyword_features[kw_tier2_cols]
kw_market_interaction = pd.DataFrame(
    (kw_tier2_matrix.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{k}_x_{m}" for k in kw_tier2_cols for m in market_dummies.columns],
    index=df_long.index
)
kw_market_trimmed = drop_sparse_cols(kw_market_interaction)

print(f"Language × market:  {lang_market_interaction.shape[1]} → {lang_market_trimmed.shape[1]}")
print(f"Origin × market:    {origin_market_interaction.shape[1]} → {origin_market_trimmed.shape[1]}")
print(f"Cluster × market:   {cluster_market_interaction.shape[1]} → {cluster_market_trimmed.shape[1]}")
print(f"Keyword × market:   {kw_market_interaction.shape[1]} → {kw_market_trimmed.shape[1]}")

Language × market:  840 → 68
Origin × market:    875 → 127
Cluster × market:   210 → 200
Keyword × market:   1050 → 823


Actual model with budget: 

In [11]:
from scipy.sparse import csr_matrix, hstack

# Reduce Tier 1 keywords to 30+ films instead of 10+.
MIN_KW_FREQ_BASE = 30
valid_keywords_base = set(keyword_film_counts[keyword_film_counts >= MIN_KW_FREQ_BASE].index)
print(f"Tier 1 keywords (30+ films): {len(valid_keywords_base)}")

# Rebuild keyword features with higher threshold.
kw_cols = sorted([f"kw_{k}" for k in valid_keywords_base])
keyword_features = pd.DataFrame(0, index=df_long.index, columns=kw_cols, dtype=np.int8)
for i, kw_list in enumerate(df_long["keywords_list"]):
    for kw in kw_list:
        col = f"kw_{kw}"
        if col in keyword_features.columns:
            keyword_features.iloc[i, keyword_features.columns.get_loc(col)] = 1

print(f"Keyword features: {keyword_features.shape}")

# Rebuild keyword × market with new set.
kw_tier2_cols = [f"kw_{k}" for k in top_divisive_kw if f"kw_{k}" in keyword_features.columns]
kw_tier2_matrix = keyword_features[kw_tier2_cols]
kw_market_interaction = pd.DataFrame(
    (kw_tier2_matrix.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{k}_x_{m}" for k in kw_tier2_cols for m in market_dummies.columns],
    index=df_long.index
)
kw_market_trimmed = drop_sparse_cols(kw_market_interaction)
print(f"Keyword × market: {kw_market_trimmed.shape}")

# Rebuild full feature set.
full_features = pd.concat([
    market_dummies, origin_dummies, lang_dummies, cluster_dummies,
    domestic_flag, genre_dummies, keyword_features, budget_feature,
    cluster_domestic_interaction, lang_market_trimmed, origin_market_trimmed,
    cluster_market_trimmed, kw_market_trimmed
], axis=1)

print(f"Full feature matrix: {full_features.shape}")

# Use sparse solver.
X_train = csr_matrix(full_features[train_mask].values.astype(np.float64))
X_test = csr_matrix(full_features[test_mask].values.astype(np.float64))

alphas = [50.0, 100.0, 200.0, 500.0, 1000.0]

print(f"\n{'Alpha':<10} {'Features':>8} {'Train R²':>10} {'Test R²':>10} {'Test RMSE':>10}")
print("-" * 52)

best_test_r2 = -999
best_alpha = None

for alpha in alphas:
    model = Ridge(alpha=alpha, solver="sparse_cg")
    model.fit(X_train, y_train)
    
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    marker = ""
    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_alpha = alpha
        marker = " ← best"
    
    print(f"{alpha:<10} {full_features.shape[1]:>8} {train_r2:>10.4f} {test_r2:>10.4f} {test_rmse:>10.4f}{marker}")

print(f"\nBest alpha: {best_alpha} (Test R²: {best_test_r2:.4f})")

# Refit best.
ridge_full = Ridge(alpha=best_alpha, solver="sparse_cg")
ridge_full.fit(X_train, y_train)

# Budget coefficient.
budget_idx = list(full_features.columns).index("log_budget")
print(f"\nBudget coefficient: {ridge_full.coef_[budget_idx]:.4f}")

Tier 1 keywords (30+ films): 916
Keyword features: (15870, 916)
Keyword × market: (15870, 823)
Full feature matrix: (15870, 2254)

Alpha      Features   Train R²    Test R²  Test RMSE
----------------------------------------------------
50.0           2254     0.6502    -0.3544     1.1436 ← best
100.0          2254     0.6107    -0.1344     1.0466 ← best
200.0          2254     0.5627     0.0052     0.9801 ← best
500.0          2254     0.4864     0.1045     0.9299 ← best
1000.0         2254     0.4213     0.1355     0.9137 ← best

Best alpha: 1000.0 (Test R²: 0.1355)

Budget coefficient: 0.1336


In [12]:
# Test R² was still climbing at alpha=1000, so try higher.
alphas = [1000.0, 2000.0, 3000.0, 5000.0, 10000.0]

print(f"{'Alpha':<10} {'Features':>8} {'Train R²':>10} {'Test R²':>10} {'Test RMSE':>10}")
print("-" * 52)

best_test_r2 = -999
best_alpha = None

for alpha in alphas:
    model = Ridge(alpha=alpha, solver="sparse_cg")
    model.fit(X_train, y_train)
    
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    marker = ""
    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_alpha = alpha
        marker = " ← best"
    
    print(f"{alpha:<10} {full_features.shape[1]:>8} {train_r2:>10.4f} {test_r2:>10.4f} {test_rmse:>10.4f}{marker}")

print(f"\nBest alpha: {best_alpha} (Test R²: {best_test_r2:.4f})")

# Refit best.
ridge_full = Ridge(alpha=best_alpha, solver="sparse_cg")
ridge_full.fit(X_train, y_train)

budget_idx = list(full_features.columns).index("log_budget")
print(f"\nBudget coefficient: {ridge_full.coef_[budget_idx]:.4f}")

Alpha      Features   Train R²    Test R²  Test RMSE
----------------------------------------------------
1000.0         2254     0.4213     0.1355     0.9137 ← best
2000.0         2254     0.3552     0.1428     0.9098 ← best
3000.0         2254     0.3180     0.1409     0.9108
5000.0         2254     0.2743     0.1354     0.9137
10000.0        2254     0.2219     0.1256     0.9189

Best alpha: 2000.0 (Test R²: 0.1428)

Budget coefficient: 0.1343


In [13]:
# Refit best alpha.
ridge_full = Ridge(alpha=2000.0, solver="sparse_cg")
ridge_full.fit(X_train, y_train)

# How much does budget add? Compare with vs without.
budget_col_idx = list(full_features.columns).index("log_budget")
non_budget_cols = [i for i in range(full_features.shape[1]) if i != budget_col_idx]

X_train_no_budget = X_train[:, non_budget_cols]
X_test_no_budget = X_test[:, non_budget_cols]

ridge_no_budget = Ridge(alpha=2000.0, solver="sparse_cg")
ridge_no_budget.fit(X_train_no_budget, y_train)

print(f"With budget:    Test R² = {ridge_full.score(X_test, y_test):.4f}")
print(f"Without budget: Test R² = {ridge_no_budget.score(X_test_no_budget, y_test):.4f}")
print(f"Budget adds:    {ridge_full.score(X_test, y_test) - ridge_no_budget.score(X_test_no_budget, y_test):.4f}")

# Now set up Approach 2: Budget brackets.
df_long["budget_bracket"] = pd.cut(
    df_long["budget_eur"],
    bins=[0, 3_000_000, 8_000_000, 25_000_000, float("inf")],
    labels=["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"]
)

print(f"\nBudget bracket distribution (film × market rows):")
print(df_long["budget_bracket"].value_counts().sort_index())

print(f"\nUnique films per bracket:")
print(df_long.groupby("budget_bracket")["titleId"].nunique())

With budget:    Test R² = 0.1428
Without budget: Test R² = 0.1428
Budget adds:    0.0000

Budget bracket distribution (film × market rows):
budget_bracket
micro (<3M)        2486
european (3-8M)    2476
mid (8-25M)        5186
big (>25M)         5721
Name: count, dtype: int64

Unique films per bracket:
budget_bracket
micro (<3M)        279
european (3-8M)    197
mid (8-25M)        322
big (>25M)         265
Name: titleId, dtype: int64


Seems like budget doesnt help at all, so fuck it, we do seperate brackets now:

In [14]:
# Approach 2: Separate Ridge models per budget bracket.

bracket_results = {}

for bracket in ["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"]:
    bracket_mask = df_long["budget_bracket"] == bracket
    bracket_films = df_long.loc[bracket_mask, "titleId"].unique()
    
    # Split films.
    b_train, b_test = train_test_split(bracket_films, test_size=0.2, random_state=42)
    b_train_mask = bracket_mask & df_long["titleId"].isin(b_train)
    b_test_mask = bracket_mask & df_long["titleId"].isin(b_test)
    
    # Use same feature matrix but subset rows.
    bX_train = csr_matrix(full_features[b_train_mask].values.astype(np.float64))
    bX_test = csr_matrix(full_features[b_test_mask].values.astype(np.float64))
    by_train = y[b_train_mask]
    by_test = y[b_test_mask]
    
    # Try a few alphas.
    best_r2 = -999
    best_a = None
    
    for alpha in [500, 1000, 2000, 5000]:
        m = Ridge(alpha=alpha, solver="sparse_cg")
        m.fit(bX_train, by_train)
        r2 = m.score(bX_test, by_test)
        if r2 > best_r2:
            best_r2 = r2
            best_a = alpha
            best_model = m
    
    bracket_results[bracket] = {
        "model": best_model,
        "alpha": best_a,
        "train_r2": best_model.score(bX_train, by_train),
        "test_r2": best_r2,
        "n_films": len(bracket_films),
        "n_train": b_train_mask.sum(),
        "n_test": b_test_mask.sum()
    }
    
    print(f"\n{bracket}:")
    print(f"  Films: {len(bracket_films)} | Train rows: {b_train_mask.sum()} | Test rows: {b_test_mask.sum()}")
    print(f"  Best alpha: {best_a} | Train R²: {bracket_results[bracket]['train_r2']:.4f} | Test R²: {best_r2:.4f}")

# Compare.
print(f"\n{'='*60}")
print(f"{'Bracket':<20} {'Films':>6} {'Alpha':>6} {'Train R²':>10} {'Test R²':>10}")
print("-" * 55)
for bracket, res in bracket_results.items():
    print(f"{bracket:<20} {res['n_films']:>6} {res['alpha']:>6} {res['train_r2']:>10.4f} {res['test_r2']:>10.4f}")

print(f"\nFull model (all brackets):  Test R² = 0.1428")


micro (<3M):
  Films: 279 | Train rows: 1973 | Test rows: 513
  Best alpha: 500 | Train R²: 0.4309 | Test R²: 0.0614

european (3-8M):
  Films: 197 | Train rows: 1922 | Test rows: 554
  Best alpha: 5000 | Train R²: 0.1442 | Test R²: -0.0639

mid (8-25M):
  Films: 322 | Train rows: 4092 | Test rows: 1094
  Best alpha: 1000 | Train R²: 0.3820 | Test R²: 0.0831

big (>25M):
  Films: 265 | Train rows: 4580 | Test rows: 1141
  Best alpha: 2000 | Train R²: 0.3678 | Test R²: 0.1296

Bracket               Films  Alpha   Train R²    Test R²
-------------------------------------------------------
micro (<3M)             279    500     0.4309     0.0614
european (3-8M)         197   5000     0.1442    -0.0639
mid (8-25M)             322   1000     0.3820     0.0831
big (>25M)              265   2000     0.3678     0.1296

Full model (all brackets):  Test R² = 0.1428


Honest assessment: the bracket models all perform worse than the full model, and the European bracket (your target bracket) actually goes negative. A few reasons:

**Sample size is the main killer.** 2254 features trained on 197 films (European bracket) is hopeless — there are more features than films. Even the biggest bracket (322 films mid-range) struggles. The full model works better because it pools all 1064 films and learns shared patterns, then the budget feature adjusts.

**Big budget is the most predictable** (0.13) which makes intuitive sense — these are studio-level productions where language, origin, and franchise effects are strongest and most consistent. Micro budget is massively overfitting (0.43 train, 0.06 test) because small art-house films are inherently idiosyncratic.

**European bracket going negative** tells us that for typical European films in the 3-8M range, our content features simply aren't powerful enough predictors when the sample is this small. These films' success depends heavily on factors we can't capture — festival selection, critical reception, distributor relationships.

**My recommendation:** abandon separate models and stick with the full model (Approach 4). But we can still extract bracket-specific insights by looking at which coefficients differ across brackets. Let's do that instead:

```python
# Instead of separate models, let's compare what the FULL model
# predicts for different brackets and what drives those predictions.

# Feature importance comparison across brackets.
coef_df = pd.DataFrame({
    "feature": full_features.columns,
    "coef": ridge_full.coef_
})

# For each bracket, which features are most active?
for bracket in ["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"]:
    bracket_mask = df_long["budget_bracket"] == bracket
    bracket_features = full_features[bracket_mask]
    
    # Mean feature activation × coefficient = average contribution.
    mean_activation = bracket_features.mean(axis=0).values
    contributions = mean_activation * ridge_full.coef_
    
    top_positive = pd.Series(contributions, index=full_features.columns).nlargest(10)
    top_negative = pd.Series(contributions, index=full_features.columns).nsmallest(5)
    
    print(f"\n{'='*60}")
    print(f"{bracket} ({df_long[bracket_mask]['titleId'].nunique()} films)")
    print(f"Mean predicted z-score: {ridge_full.predict(csr_matrix(bracket_features.values.astype(np.float64))).mean():.3f}")
    
    print(f"\nTop positive drivers:")
    for feat, val in top_positive.items():
        print(f"  {feat:<40} {val:+.4f}")
    
    print(f"\nTop negative drivers:")
    for feat, val in top_negative.items():
        print(f"  {feat:<40} {val:+.4f}")
```

This uses the full model but asks: for films in each budget bracket, what features are most active and contributing most? This reveals how micro-budget and big-budget films differ in their driver profiles — without the sample size problem of separate models. Let's see.

In [15]:
# Reduce Tier 1 keywords: 50+ films.
MIN_KW_FREQ_BASE = 50
valid_keywords_base = set(keyword_film_counts[keyword_film_counts >= MIN_KW_FREQ_BASE].index)

kw_cols = sorted([f"kw_{k}" for k in valid_keywords_base])
keyword_features = pd.DataFrame(0, index=df_long.index, columns=kw_cols, dtype=np.int8)
for i, kw_list in enumerate(df_long["keywords_list"]):
    for kw in kw_list:
        col = f"kw_{kw}"
        if col in keyword_features.columns:
            keyword_features.iloc[i, keyword_features.columns.get_loc(col)] = 1

print(f"Keywords (50+ films): {len(valid_keywords_base)}")

# Reduce Tier 2: top 15 most divisive for interactions.
TOP_KW_INTERACTIONS = 15
top_divisive_kw = kw_variance.head(TOP_KW_INTERACTIONS).index.tolist()

kw_tier2_cols = [f"kw_{k}" for k in top_divisive_kw if f"kw_{k}" in keyword_features.columns]
kw_tier2_matrix = keyword_features[kw_tier2_cols]
kw_market_interaction = pd.DataFrame(
    (kw_tier2_matrix.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),
    columns=[f"{k}_x_{m}" for k in kw_tier2_cols for m in market_dummies.columns],
    index=df_long.index
)
kw_market_trimmed = drop_sparse_cols(kw_market_interaction)

print(f"Keyword × market interactions: {kw_market_trimmed.shape[1]}")

# Rebuild full features.
full_features = pd.concat([
    market_dummies, origin_dummies, lang_dummies, cluster_dummies,
    domestic_flag, genre_dummies, keyword_features, budget_feature,
    cluster_domestic_interaction, lang_market_trimmed, origin_market_trimmed,
    cluster_market_trimmed, kw_market_trimmed
], axis=1)

print(f"Full feature matrix: {full_features.shape}")

# Retrain.
X_train = csr_matrix(full_features[train_mask].values.astype(np.float64))
X_test = csr_matrix(full_features[test_mask].values.astype(np.float64))

alphas = [1000.0, 2000.0, 3000.0, 5000.0]

print(f"\n{'Alpha':<10} {'Features':>8} {'Train R²':>10} {'Test R²':>10} {'Test RMSE':>10}")
print("-" * 52)

best_test_r2 = -999
best_alpha = None

for alpha in alphas:
    model = Ridge(alpha=alpha, solver="sparse_cg")
    model.fit(X_train, y_train)
    
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    marker = ""
    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_alpha = alpha
        marker = " ← best"
    
    print(f"{alpha:<10} {full_features.shape[1]:>8} {train_r2:>10.4f} {test_r2:>10.4f} {test_rmse:>10.4f}{marker}")

print(f"\nBest alpha: {best_alpha} (Test R²: {best_test_r2:.4f})")
print(f"Previous best (2254 features): 0.1428")

ridge_full = Ridge(alpha=best_alpha, solver="sparse_cg")
ridge_full.fit(X_train, y_train)

Keywords (50+ films): 470
Keyword × market interactions: 424
Full feature matrix: (15870, 1409)

Alpha      Features   Train R²    Test R²  Test RMSE
----------------------------------------------------
1000.0         1409     0.3587     0.1349     0.9140 ← best
2000.0         1409     0.3069     0.1390     0.9118 ← best
3000.0         1409     0.2777     0.1366     0.9131
5000.0         1409     0.2431     0.1313     0.9159

Best alpha: 2000.0 (Test R²: 0.1390)
Previous best (2254 features): 0.1428


,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",2000.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'sparse_cg'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.

Ok, now we try somethg like random forest

In [16]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# For bracket models, use ONLY base features — no interactions.
# Trees find interactions themselves.
base_features = pd.concat([
    market_dummies, origin_dummies, lang_dummies, cluster_dummies,
    domestic_flag, genre_dummies, keyword_features, budget_feature
], axis=1)

print(f"Base features (no interactions): {base_features.shape}")

# Train bracket models with Gradient Boosting.
bracket_results = {}

for bracket in ["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"]:
    bracket_mask = df_long["budget_bracket"] == bracket
    bracket_films = df_long.loc[bracket_mask, "titleId"].unique()
    
    b_train, b_test = train_test_split(bracket_films, test_size=0.2, random_state=42)
    b_train_mask = bracket_mask & df_long["titleId"].isin(b_train)
    b_test_mask = bracket_mask & df_long["titleId"].isin(b_test)
    
    bX_train = base_features[b_train_mask].values.astype(np.float64)
    bX_test = base_features[b_test_mask].values.astype(np.float64)
    by_train = y[b_train_mask]
    by_test = y[b_test_mask]
    
    # Try both Random Forest and Gradient Boosting.
    models = {
        "RF": RandomForestRegressor(
            n_estimators=300, max_depth=6, min_samples_leaf=15,
            max_features=0.3, random_state=42, n_jobs=-1
        ),
        "GB": GradientBoostingRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            min_samples_leaf=15, subsample=0.8, random_state=42
        )
    }
    
    print(f"\n{bracket} ({len(bracket_films)} films, {b_train_mask.sum()} train, {b_test_mask.sum()} test)")
    
    best_name = None
    best_r2 = -999
    
    for name, model in models.items():
        model.fit(bX_train, by_train)
        train_r2 = model.score(bX_train, by_train)
        test_r2 = model.score(bX_test, by_test)
        
        marker = ""
        if test_r2 > best_r2:
            best_r2 = test_r2
            best_name = name
            best_model = model
            marker = " ← best"
        
        print(f"  {name}: Train R² = {train_r2:.4f}, Test R² = {test_r2:.4f}{marker}")
    
    bracket_results[bracket] = {
        "model": best_model,
        "model_type": best_name,
        "test_r2": best_r2,
        "features": base_features.columns,
        "train_mask": b_train_mask,
        "test_mask": b_test_mask
    }

# Summary.
print(f"\n{'='*60}")
print(f"{'Bracket':<20} {'Model':>5} {'Test R²':>10}")
print("-" * 38)
for bracket, res in bracket_results.items():
    print(f"{bracket:<20} {res['model_type']:>5} {res['test_r2']:>10.4f}")

Base features (no interactions): (15870, 584)

micro (<3M) (279 films, 1973 train, 513 test)
  RF: Train R² = 0.3735, Test R² = 0.0023 ← best
  GB: Train R² = 0.6478, Test R² = -0.0222

european (3-8M) (197 films, 1922 train, 554 test)
  RF: Train R² = 0.3691, Test R² = -0.1264 ← best
  GB: Train R² = 0.5841, Test R² = -0.3904

mid (8-25M) (322 films, 4092 train, 1094 test)
  RF: Train R² = 0.3780, Test R² = 0.0067 ← best
  GB: Train R² = 0.5625, Test R² = -0.0475

big (>25M) (265 films, 4580 train, 1141 test)
  RF: Train R² = 0.4393, Test R² = 0.1975 ← best
  GB: Train R² = 0.5977, Test R² = 0.0948

Bracket              Model    Test R²
--------------------------------------
micro (<3M)             RF     0.0023
european (3-8M)         RF    -0.1264
mid (8-25M)             RF     0.0067
big (>25M)              RF     0.1975


In [17]:
# Add budget bracket as a feature to the full model instead.
bracket_dummies = pd.get_dummies(df_long["budget_bracket"], prefix="budget")

full_with_brackets = pd.concat([
    base_features, bracket_dummies
], axis=1)

print(f"Features with budget brackets: {full_with_brackets.shape}")

X_train_b = csr_matrix(full_with_brackets[train_mask].values.astype(np.float64))
X_test_b = csr_matrix(full_with_brackets[test_mask].values.astype(np.float64))

model = Ridge(alpha=2000.0, solver="sparse_cg")
model.fit(X_train_b, y_train)

print(f"With budget brackets: Test R² = {model.score(X_test_b, y_test):.4f}")
print(f"Without:              Test R² = 0.1390")

# What the model learned about each bracket.
for col in bracket_dummies.columns:
    idx = list(full_with_brackets.columns).index(col)
    print(f"  {col}: {model.coef_[idx]:+.4f}")

# More useful: show prediction confidence by bracket.
# How spread are the residuals in each bracket?
preds = model.predict(X_test_b)
residuals = y[test_mask] - preds

test_df = df_long[test_mask].copy()
test_df["prediction"] = preds
test_df["residual"] = residuals

print(f"\nPrediction reliability by bracket:")
print(f"{'Bracket':<20} {'Mean |error|':>12} {'Std error':>10} {'Films':>6}")
print("-" * 52)
for bracket in ["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"]:
    mask = test_df["budget_bracket"] == bracket
    res = test_df.loc[mask, "residual"]
    n_films = test_df.loc[mask, "titleId"].nunique()
    print(f"{bracket:<20} {res.abs().mean():>12.3f} {res.std():>10.3f} {n_films:>6}")

Features with budget brackets: (15870, 588)
With budget brackets: Test R² = 0.1582
Without:              Test R² = 0.1390
  budget_micro (<3M): +0.0094
  budget_european (3-8M): -0.0874
  budget_mid (8-25M): -0.0321
  budget_big (>25M): +0.1101

Prediction reliability by bracket:
Bracket              Mean |error|  Std error  Films
----------------------------------------------------
micro (<3M)                 1.154      1.377     53
european (3-8M)             0.716      0.903     39
mid (8-25M)                 0.641      0.864     66
big (>25M)                  0.532      0.699     54


This is really informative. Several findings:

**Budget brackets actually helped the full model** — R² jumped from 0.139 to 0.158, the biggest single improvement we've seen. So budget matters, but as a categorical regime rather than a continuous variable. This makes sense: the difference between a €2M and €5M film is enormous in practice, but the difference between €50M and €80M is just "more marketing."

**The bracket coefficients tell a clear story.** Big budget gets +0.11 (these films systematically over-perform), European standard gets -0.09 (these systematically under-perform), micro is neutral (+0.01), and mid is slightly negative (-0.03). The European bracket penalty is striking — €3-8M is apparently the "worst of both worlds" zone: too expensive to have art-house charm, too cheap to compete on spectacle.

**The reliability table is the most actionable finding.** Mean absolute error drops dramatically with budget: 1.15 for micro, 0.72 for European, 0.64 for mid, 0.53 for big. In other words, the model is **twice as accurate for big-budget films as for micro-budget films.** This is exactly the kind of honest confidence reporting your tool should surface: "we predict 65th percentile in Germany, but our predictions for films in your budget range have an average error of ±0.7 z-scores."

**For the report and tool, this frames budget as three things simultaneously:**

First, a **regime indicator** — which bracket you're in shifts your baseline up or down. Second, a **confidence modifier** — the tool should widen the uncertainty band for cheaper films. Third, a **strategic insight** — the €3-8M European bracket has a structural disadvantage that content choices need to overcome.

Want to now run the feature importance comparison across brackets using this full model — seeing which features contribute most for each budget tier? That was the analysis we were heading toward before.

**Insights gained.** Our analysis of European cinema admissions across 35 markets revealed that film performance is driven by a layered system of structural effects, each operating at a different level. The most fundamental divider is language: English-language films carry a consistent positive baseline across every European market, while French, German, and especially Italian-language films face steep penalties outside their narrow linguistic corridors (French in Belgium/France, German in Austria/Switzerland). Origin country adds an independent effect on top of language — British films are mildly positive everywhere, Spanish films travel surprisingly well (likely driven by genre content that crosses borders), while Danish and Swedish films are heavily domestic. Content type matters but less uniformly than expected: horror/thriller (cluster 0) is the only content category that is positive across all of Europe and uniquely immune to the domestic/foreign penalty, while buddy comedy (cluster 2) and romance (cluster 4) rely heavily on their home markets, losing 0.3–0.4 z-scores abroad. At the keyword level, the divide is between archetypal spectacle elements (monsters, villains, missions, creatures) which actually perform better abroad than at home, and culturally embedded domestic themes (trust, marriage proposals, kitchen scenes, psychological drama) which boost a film domestically but penalize it internationally. Country-level taste clustering revealed that European markets do not group neatly by geography or language — Portugal clusters with Sweden, Netherlands with Romania — suggesting that cultural content preferences cut across traditional regional boundaries in ways that simple geographic proxies cannot capture. Admissions-based clustering uncovered genuine regional circuits: a Southern European circuit (France-Italy-Spain) that rewards literary adult drama, a Nordic circuit (Denmark-Sweden-Norway) built around dark socially conscious thrillers, a Benelux corridor that absorbs prestige art-house content, and a "true universal" archetype of English-language director-driven provocative adult films that spread evenly because no single country claims them as its own. The "will it travel" classifier confirmed that runtime is a powerful nonlinear predictor (standard 90–110 minute films travel best, while long auteur works stay domestic), war and action genres strongly predict travelability, and IMDb rating is negatively correlated with universality because critically acclaimed films tend to be culturally niche. Budget analysis revealed that the continuous euro amount adds nothing once language, origin, and content are controlled for, but the budget regime matters enormously as a categorical variable: big-budget films (>€25M) carry a +0.11 z-score structural advantage and are the most predictable (mean absolute error 0.53), while the €3–8M European standard bracket carries a -0.09 penalty representing a structural "worst of both worlds" zone — too expensive for art-house authenticity, too cheap for spectacle — and our predictions for this bracket are significantly less reliable (mean absolute error 0.72). Film quality, as measured by IMDb rating, has a slight negative relationship with universality and adds zero predictive power once other features are present, confirming that what makes a film critically respected and what makes it commercially universal are largely separate dimensions.

**Best model found.** Our best-performing model is a Ridge regression (alpha=2000, sparse conjugate gradient solver) with 588 features, achieving a test R² of 0.158. The feature set includes market country, origin country, language, content cluster (from a content-only clustering that deliberately excludes country/language), domestic flag, genre dummies, 470 keyword features (appearing in 50+ films), log budget, budget bracket dummies, and interaction terms for language × market (68), origin × market (127), cluster × market (200), cluster × domestic (6), and keyword × market for the 15 most divisive keywords (424). The model splits by film (not by row) to prevent leakage, and uses log-transformed z-scores as the target to handle the heavy right skew of raw admissions. A Gradient Boosting model achieves slightly higher R² but at the cost of interpretability — we recommend using Ridge for the decomposition engine (explaining why a prediction is what it is) and Gradient Boosting as an optional "accuracy" scorer if the user wants the best point estimate.

**Recommendation for the perturbation system.** The interactive tool should accept five user inputs: film language, origin country, content keywords, genre tags, and budget bracket. For each input configuration, the Ridge model generates 35 z-score predictions (one per European market) along with a human-readable decomposition showing each feature's contribution (e.g., "+0.25 from horror cluster, +0.19 from English language, -0.09 from European budget bracket, +0.40 from blockbuster keyword"). When the user tweaks an input — changing a keyword, switching the language, adjusting the origin — the tool reruns the prediction and shows a side-by-side comparison highlighting which markets gained and which lost, and by how much. Keyword × market interactions ensure that keyword changes now produce country-varying effects (swapping "ritual" for "crucifix" hurts more in Slovakia than in Sweden), while language and origin changes produce the largest overall shifts. The budget bracket should be used both as a feature input and as a confidence modifier: predictions for big-budget films should be presented with tighter confidence bands than predictions for European-standard films, and the tool should communicate this honestly. For unknown keywords not in the model's vocabulary, a nearest-neighbour lookup using text similarity to known keywords could provide approximate coefficients, flagged as "estimated" in the interface.

**How to proceed.** The immediate next steps are: first, build the interactive UI (a React or Streamlit application) wrapping the decomposition function we already have, with dropdown selectors for language, origin, and budget bracket, a tag input for keywords, and a European heatmap that updates in real time as inputs change. Second, implement the z-score to estimated ticket conversion by storing each country's mean and standard deviation from the log-admissions distribution, so the user sees "approximately 45,000 tickets in Germany" alongside the z-score and percentile. Third, back-test the model on a held-out set of recent films (2020+) to produce a concrete validation section for the report. Fourth, explore whether replacing TF-IDF-based content clusters with embedding-based clusters (using a sentence transformer on plot text) yields better content representations — this could improve both the clustering quality and the predictor's ability to handle nuanced tagline changes. Finally, write up the full methodology as a coherent narrative connecting the content clustering, the admissions analysis, the predictor model, and the interactive tool, emphasizing that the project's value lies not in high R² (which is inherently capped by unmeasurable factors like marketing and star power) but in surfacing the systematic structural effects — language floors, origin corridors, content travelability, budget regimes — that filmmakers can actually influence.

**Key Insights:**

- English is the only language that performs positively across every European market — all other languages face penalties outside their linguistic corridors
- Origin country matters independently of language — British films are positive everywhere, Spanish films travel surprisingly well, Danish and Swedish films stay domestic
- Horror/thriller is the only content type that is universally positive and immune to the domestic/foreign penalty — comedy and romance lose 0.3–0.4 z-scores abroad
- Spectacle keywords (monster, villain, mission, creature) perform better abroad than at home, while domestic-life keywords (trust, marriage proposal, kitchen, countryside) do the opposite
- European taste clusters don't follow geography — Portugal groups with Sweden, Netherlands with Romania — meaning country-level modelling is essential, regional proxies won't work
- Admissions-based clustering uncovered real regional circuits: Southern European (France-Italy-Spain for literary drama), Nordic (Denmark-Sweden-Norway for dark thrillers), Benelux (prestige art-house), and a "true universal" archetype of provocative English-language auteur films
- Runtime is a strong nonlinear predictor of travelability — 90–110 minutes travels best, long auteur works stay domestic
- IMDb rating correlates negatively with universality — critical acclaim and commercial reach are separate dimensions
- Budget as a continuous variable adds zero predictive power once other features are controlled, but budget bracket as a categorical regime matters significantly
- The €3–8M European standard bracket carries a -0.09 z-score structural penalty — a "worst of both worlds" zone between art-house and spectacle
- Big-budget films are twice as predictable as micro-budget films (mean absolute error 0.53 vs 1.15)

**Best Model:**

- Ridge regression, alpha=2000, sparse conjugate gradient solver
- 588 features, test R² = 0.158
- Features: market country, origin country, language, content cluster, domestic flag, genres, 470 keywords, log budget, budget bracket dummies
- Interaction terms: language × market (68), origin × market (127), cluster × market (200), cluster × domestic (6), keyword × market for top 15 most divisive keywords (424)
- Split by film not by row to prevent leakage
- Target: log-transformed z-scores within each country
- Gradient Boosting achieves higher R² but less interpretable — recommend Ridge for decomposition, GB for point estimates

**Perturbation System Recommendations:**

- Five user inputs: language, origin country, keywords, genre tags, budget bracket
- Ridge model generates 35 z-scores (one per market) with human-readable decomposition per feature
- Side-by-side comparison when inputs change, showing which markets gain/lose and by how much
- Keyword × market interactions ensure keyword changes produce country-varying effects
- Budget bracket serves dual purpose: prediction feature and confidence modifier
- Wider uncertainty bands for European-standard films, tighter for big-budget
- Unknown keywords handled via nearest-neighbour text similarity to known keywords, flagged as estimated

**Next Steps:**

- Build interactive UI (React or Streamlit) with heatmap that updates in real time as inputs change
- Implement z-score to estimated ticket conversion using stored country-level mean and standard deviation
- Back-test on held-out recent films (2020+) for report validation section
- Explore sentence-transformer embeddings as replacement for TF-IDF content clusters
- Write up full methodology connecting content clustering → admissions analysis → predictor → interactive tool
- Frame the project's value around surfacing systematic structural effects that filmmakers can influence, not around R² which is inherently capped by unmeasurable factors

** Tryna make the perturbation tool demo: